# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asiya-Akhtar/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one pseudonymized client, one pseudonymized content item, and one report date.

The analysis window is March 2026 (month='2026-03').

I use a mid-panel month because the final month is reserved as a sealed test period.

In [25]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""")

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- impressions: number of times content was shown
- clicks: number of clicks received
- ctr: click-through rate
- average_position: search ranking position
- device: user device category

### Label
- Future content performance outcome (future clicks or ranking proxy)

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
- Future clicks: excluded because it is not available at prediction time and would cause leakage.
- Raw identifiers: excluded because they are pseudonymized and not useful as prediction features.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Query 1: Grain Check

In [28]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ duplicate_rows │
│    date     │    varchar     │     varchar     │     int64      │
├─────────────┴────────────────┴─────────────────┴────────────────┤
│                             0 rows                              │
└─────────────────────────────────────────────────────────────────┘

Query 2: Row Count + Window

In [29]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
""")

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

Query 3: Availability (IS TRUE)

This dataset cannot reveal real client identities because identifiers are pseudonymized.

The history is unbalanced because different clients have different available data periods.

This analysis uses March 2026 only, so it does not measure seasonal patterns across the full year.

The results are directional and intended for decision support, not causal conclusions.

In [30]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [31]:
con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2026-03'
AND gsc_data_available IS TRUE
""")

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘

## Data limits

This dataset cannot reveal real client identities because identifiers are pseudonymized.

The history is unbalanced because different clients have different available data periods.

This analysis uses March 2026 only, so seasonal patterns across the full year are not measured.

The results are directional and intended for decision support, not causal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.